# Machine Learning Benchmark: 12 Algorithms Across Classification, Boosting, and Clustering

This notebook implements, evaluates, and compares **12 Machine Learning algorithms** using real-world datasets:
1. **Classification (4 Algorithms)**: Decision Tree, Random Forest, K-Nearest Neighbors (KNN), Support Vector Machine (SVM) on *Social Network Ads*
2. **Boosting (4 Algorithms)**: AdaBoost, Gradient Boosting, XGBoost, HistGradientBoosting on *Insurance Dataset*
3. **Clustering (4 Algorithms)**: K-Means, Agglomerative Hierarchical, DBSCAN, Gaussian Mixture Model (GMM) on *Mall Customers*

---
### Quick Setup Instructions for VS Code & Google Colab
- **VS Code**: Install the Python and Jupyter extensions. Open this notebook and choose your Python environment.
- **Google Colab**: Upload this notebook along with `social_networks.csv`, `insurance.csv`, and `mall_customers.csv` to your session storage.


In [ ]:
# Cell 1: Dependency Installation
!pip install --quiet xgboost seaborn scikit-learn pandas numpy matplotlib joblib


In [ ]:
# Cell 2: Imports & Configuration
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
import scipy.cluster.hierarchy as sch

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("plots", exist_ok=True)
print("All libraries successfully imported!")


## Part 1: Classification (4 Algorithms) on Social Network Ads
**Goal**: Predict whether a user purchased a product (`Purchased`: 0 or 1) based on their `Age`, `EstimatedSalary`, and `Gender`.


In [ ]:
# Step 1.1: Load & Explore Social Networks Dataset
df_soc = pd.read_csv("social_networks.csv")
print(f"Dataset Shape: {df_soc.shape[0]} rows, {df_soc.shape[1]} columns")
print("\nMissing values:\n", df_soc.isnull().sum())
print("\nClass Distribution:\n", df_soc["Purchased"].value_counts())
display(df_soc.head())


In [ ]:
# Step 1.2: Preprocessing & Stratified Train/Test Split
le_gender = LabelEncoder()
df_soc["Gender_Encoded"] = le_gender.fit_transform(df_soc["Gender"])

X_soc = df_soc[["Age", "EstimatedSalary", "Gender_Encoded"]].copy()
y_soc = df_soc["Purchased"].copy()

X_soc_train, X_soc_test, y_soc_train, y_soc_test = train_test_split(
    X_soc, y_soc, test_size=0.20, random_state=42, stratify=y_soc
)

scaler_soc = StandardScaler()
X_soc_train_scaled = scaler_soc.fit_transform(X_soc_train)
X_soc_test_scaled = scaler_soc.transform(X_soc_test)

print(f"Train size: {X_soc_train.shape[0]} | Test size: {X_soc_test.shape[0]}")


In [ ]:
# Step 1.3: Train & Evaluate 4 Classification Algorithms
clf_models = {
    "Decision Tree": (DecisionTreeClassifier(criterion="entropy", random_state=42), False),
    "Random Forest": (RandomForestClassifier(n_estimators=100, criterion="entropy", random_state=42), False),
    "K-Nearest Neighbors (KNN)": (KNeighborsClassifier(n_neighbors=5), True),
    "Support Vector Machine (SVM)": (SVC(kernel="rbf", C=1.0, random_state=42), True)
}

clf_results = []
clf_cms = {}

for name, (model, is_scaled) in clf_models.items():
    X_tr = X_soc_train_scaled if is_scaled else X_soc_train
    X_te = X_soc_test_scaled if is_scaled else X_soc_test
    
    model.fit(X_tr, y_soc_train)
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    tr_acc = accuracy_score(y_soc_train, y_tr_pred)
    te_acc = accuracy_score(y_soc_test, y_te_pred)
    prec = precision_score(y_soc_test, y_te_pred, zero_division=0)
    rec = recall_score(y_soc_test, y_te_pred, zero_division=0)
    f1 = f1_score(y_soc_test, y_te_pred, zero_division=0)
    
    clf_cms[name] = confusion_matrix(y_soc_test, y_te_pred)
    clf_results.append({
        "Algorithm": name,
        "Train Accuracy (%)": round(tr_acc * 100, 2),
        "Test Accuracy (%)": round(te_acc * 100, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

df_clf_res = pd.DataFrame(clf_results)
print("=== CLASSIFICATION EVALUATION METRICS ===")
display(df_clf_res)


In [ ]:
# Step 1.4: Confusion Matrices Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, (name, cm) in enumerate(clf_cms.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                annot_kws={"size": 14, "weight": "bold"},
                xticklabels=["Not Purchased (0)", "Purchased (1)"],
                yticklabels=["Not Purchased (0)", "Purchased (1)"])
    acc_val = df_clf_res.loc[df_clf_res["Algorithm"] == name, "Test Accuracy (%)"].values[0]
    ax.set_title(f"{name}\nTest Accuracy: {acc_val:.1f}%", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
plt.suptitle("Confusion Matrices — Classification Algorithms", fontsize=15, weight="bold", y=0.99)
plt.tight_layout()
plt.show()


## Part 2: Boosting (4 Algorithms) on Insurance Dataset
**Goal**: Predict `smoker` status (1 for Yes, 0 for No) using demographic and medical charges data.


In [ ]:
# Step 2.1: Load & Preprocess Insurance Dataset
df_ins = pd.read_csv("insurance.csv")
print(f"Insurance Shape: {df_ins.shape[0]} rows, {df_ins.shape[1]} columns")
display(df_ins.head())

le_smoker = LabelEncoder()
df_ins["smoker_encoded"] = le_smoker.fit_transform(df_ins["smoker"])
le_sex = LabelEncoder()
df_ins["sex_encoded"] = le_sex.fit_transform(df_ins["sex"])

feature_cols_ins = ["age", "sex_encoded", "bmi", "children", "charges"]
X_ins = df_ins[feature_cols_ins].copy()
y_ins = df_ins["smoker_encoded"].copy()

X_ins_train, X_ins_test, y_ins_train, y_ins_test = train_test_split(
    X_ins, y_ins, test_size=0.20, random_state=42, stratify=y_ins
)
print(f"Train size: {X_ins_train.shape[0]} | Test size: {X_ins_test.shape[0]}")


In [ ]:
# Step 2.2: Train & Evaluate 4 Boosting Algorithms
boost_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42, eval_metric="logloss"),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_depth=6, random_state=42)
}

boost_results = []
boost_cms = {}

for name, model in boost_models.items():
    model.fit(X_ins_train, y_ins_train)
    y_tr_pred = model.predict(X_ins_train)
    y_te_pred = model.predict(X_ins_test)
    y_te_proba = model.predict_proba(X_ins_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    tr_acc = accuracy_score(y_ins_train, y_tr_pred)
    te_acc = accuracy_score(y_ins_test, y_te_pred)
    prec = precision_score(y_ins_test, y_te_pred, zero_division=0)
    rec = recall_score(y_ins_test, y_te_pred, zero_division=0)
    f1 = f1_score(y_ins_test, y_te_pred, zero_division=0)
    roc_auc = roc_auc_score(y_ins_test, y_te_proba) if y_te_proba is not None else np.nan
    
    boost_cms[name] = confusion_matrix(y_ins_test, y_te_pred)
    boost_results.append({
        "Algorithm": name,
        "Train Accuracy (%)": round(tr_acc * 100, 2),
        "Test Accuracy (%)": round(te_acc * 100, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4)
    })

df_boost_res = pd.DataFrame(boost_results)
print("=== BOOSTING EVALUATION METRICS ===")
display(df_boost_res)


In [ ]:
# Step 2.3: Confusion Matrices for Boosting
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, (name, cm) in enumerate(boost_cms.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False, ax=ax,
                annot_kws={"size": 14, "weight": "bold"},
                xticklabels=["Non-Smoker (0)", "Smoker (1)"],
                yticklabels=["Non-Smoker (0)", "Smoker (1)"])
    acc_val = df_boost_res.loc[df_boost_res["Algorithm"] == name, "Test Accuracy (%)"].values[0]
    ax.set_title(f"{name}\nTest Accuracy: {acc_val:.1f}%", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
plt.suptitle("Confusion Matrices — Boosting Algorithms (Insurance)", fontsize=15, weight="bold", y=0.99)
plt.tight_layout()
plt.show()


## Part 3: Clustering (4 Algorithms) on Mall Customers
**Goal**: Discover customer market segments based on `Annual Income (k$)` and `Spending Score (1-100)` without using any labels.


In [ ]:
# Step 3.1: Load & Scale Mall Customers Dataset
df_mall = pd.read_csv("mall_customers.csv")
print(f"Mall Customers Shape: {df_mall.shape[0]} rows, {df_mall.shape[1]} columns")
display(df_mall.head())

X_mall = df_mall[["Annual Income (k$)", "Spending Score (1-100)"]].copy()
scaler_mall = StandardScaler()
X_mall_scaled = scaler_mall.fit_transform(X_mall)

# Elbow Method & Silhouette
inertias = []
sils = []
for k in range(2, 11):
    km_t = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_mall_scaled)
    inertias.append(km_t.inertia_)
    sils.append(silhouette_score(X_mall_scaled, km_t.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(range(2, 11), inertias, "ro-", linewidth=2)
ax1.set_title("Elbow Method for Optimal K", weight="bold")
ax1.set_xlabel("K")
ax1.set_ylabel("Inertia")
ax1.axvline(5, color="gray", linestyle="--")

ax2.plot(range(2, 11), sils, "bs-", linewidth=2)
ax2.set_title("Silhouette Score vs K", weight="bold")
ax2.set_xlabel("K")
ax2.set_ylabel("Silhouette Score")
ax2.axvline(5, color="green", linestyle="--")
plt.tight_layout()
plt.show()


In [ ]:
# Step 3.2: Train & Evaluate 4 Clustering Algorithms
k_opt = 5
km = KMeans(n_clusters=k_opt, init="k-means++", random_state=42, n_init=10)
lbl_km = km.fit_predict(X_mall_scaled)

agg = AgglomerativeClustering(n_clusters=k_opt, linkage="ward")
lbl_agg = agg.fit_predict(X_mall_scaled)

db = DBSCAN(eps=0.35, min_samples=5)
lbl_db = db.fit_predict(X_mall_scaled)

gmm = GaussianMixture(n_components=k_opt, covariance_type="full", random_state=42)
lbl_gmm = gmm.fit_predict(X_mall_scaled)

clust_dict = {
    "K-Means": lbl_km,
    "Agglomerative Hierarchical": lbl_agg,
    "DBSCAN": lbl_db,
    "Gaussian Mixture Model (GMM)": lbl_gmm
}

clust_results = []
for name, labels in clust_dict.items():
    uniq = set(labels)
    n_cl = len(uniq - {-1})
    n_noise = list(labels).count(-1)
    if n_cl > 1:
        if -1 in uniq:
            m = (labels != -1)
            sil = silhouette_score(X_mall_scaled[m], labels[m])
            db_i = davies_bouldin_score(X_mall_scaled[m], labels[m])
            ch_i = calinski_harabasz_score(X_mall_scaled[m], labels[m])
        else:
            sil = silhouette_score(X_mall_scaled, labels)
            db_i = davies_bouldin_score(X_mall_scaled, labels)
            ch_i = calinski_harabasz_score(X_mall_scaled, labels)
    else:
        sil, db_i, ch_i = np.nan, np.nan, np.nan
        
    clust_results.append({
        "Algorithm": name,
        "Clusters Found": n_cl,
        "Noise Points": n_noise,
        "Silhouette Score": round(sil, 4),
        "Davies-Bouldin Index": round(db_i, 4),
        "Calinski-Harabasz Index": round(ch_i, 2)
    })

df_clust_res = pd.DataFrame(clust_results)
print("=== CLUSTERING EVALUATION METRICS ===")
display(df_clust_res)


In [ ]:
# Step 3.3: 2D Cluster Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()
colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12", "#1abc9c"]

for idx, (name, labels) in enumerate(clust_dict.items()):
    ax = axes[idx]
    for lbl in sorted(list(set(labels))):
        if lbl == -1:
            mask = (labels == lbl)
            ax.scatter(X_mall.loc[mask, "Annual Income (k$)"],
                       X_mall.loc[mask, "Spending Score (1-100)"],
                       c="gray", marker="x", s=50, label="Noise")
        else:
            mask = (labels == lbl)
            ax.scatter(X_mall.loc[mask, "Annual Income (k$)"],
                       X_mall.loc[mask, "Spending Score (1-100)"],
                       c=colors[lbl % len(colors)], s=55, alpha=0.85, edgecolors="k", label=f"Cluster {lbl+1}")
    if name == "K-Means":
        cents = scaler_mall.inverse_transform(km.cluster_centers_)
        ax.scatter(cents[:, 0], cents[:, 1], s=200, c="yellow", marker="*", edgecolors="k", label="Centroids")
    ax.set_title(f"{name}", weight="bold")
    ax.set_xlabel("Annual Income (k$)")
    ax.set_ylabel("Spending Score (1-100)")
    ax.legend(fontsize=8, loc="upper right")
plt.suptitle("Customer Segments Identified by 4 Clustering Algorithms", fontsize=15, weight="bold", y=0.99)
plt.tight_layout()
plt.show()


## Part 4: Grand Master Comparison & 100% Accuracy Milestones
Here we compare the performance of all supervised learning algorithms and review the 100% training fidelity and out-of-sample generalization.


In [ ]:
# Step 4.1: Grand Comparison Plot of Supervised Algorithms
df_sup = pd.concat([df_clf_res, df_boost_res], ignore_index=True)

fig, ax = plt.subplots(figsize=(14, 6))
idx = np.arange(len(df_sup))
w = 0.38

b1 = ax.bar(idx - w/2, df_sup["Train Accuracy (%)"], w, label="Train Accuracy (%)", color="#2ecc71", edgecolor="k")
b2 = ax.bar(idx + w/2, df_sup["Test Accuracy (%)"], w, label="Test Accuracy (%)", color="#3498db", edgecolor="k")

for b in b1:
    h = b.get_height()
    ax.annotate(f"{h:.1f}%", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4), textcoords="offset points",
                ha="center", weight="bold", fontsize=9, color="darkgreen" if h >= 99.9 else "black")
for b in b2:
    h = b.get_height()
    ax.annotate(f"{h:.1f}%", xy=(b.get_x() + b.get_width()/2, h), xytext=(0, 4), textcoords="offset points",
                ha="center", weight="bold", fontsize=9)

ax.set_xticks(idx)
ax.set_xticklabels(df_sup["Algorithm"], rotation=20, ha="right", fontsize=10)
ax.set_ylim(70, 110)
ax.axhline(100, color="red", linestyle="--", label="100% Accuracy Ceiling")
ax.set_ylabel("Accuracy (%)", weight="bold")
ax.set_title("Train vs Test Accuracy Across All 8 Supervised Algorithms", weight="bold", fontsize=14)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
